## Provider Anomaly: LLM Case Notes (v4.1, GPT-OSS)

Writes a short case note for the referred providers that matter most in each claim category, and a deterministic note for every other referred provider. Every fact and number in a note comes from Python; the LLM only puts them into sentences.

### How a provider is handled
1. **Referred providers only.** The provider list and the provider anomaly score come from the referral file saved by `combine_all_projects_anomaly_predictions` (same business filters as the referral dashboard).
2. **Features from the exploded importance file** (`explained_top_features_{scoring_date}.csv`): one row per feature is kept (its strongest analysis level), with the levels that agree.
3. **Every feature has a dictionary row** (`feature_mapping_v2_3.csv`); the run stops otherwise. Each row carries a Definition_Confidence; level-dependent measures say which analysis level they come from.
4. **Patient mix:** when patient demographics make up at least half of the provider's top signal and no billing measure is among its 3 strongest, the provider gets a fixed patient-mix note and goes on the model-review list. No LLM call.
5. **Main patterns:** up to 3 billing measures inside the provider's 80% signal coverage (Importance_Tier top_50 or top_80), in Feature_Rank_The_Provider order. A measure on the harmless side of peers becomes background.
6. **Narrative selection:** per category, providers at or above the 90th percentile of the provider anomaly score among referred providers, with at least 5 per category. Everyone else gets a deterministic evidence-only note.
7. **Numbers from Python:** provider value, peer median and percentile, only where the scoring files hold the measure unambiguously.
8. **Schemes from the dictionary:** a scheme is confirmed only when a main pattern links to it and the provider's measured value is on the concerning side of peers. A main pattern with no scheme link gives "No known scheme (possible new pattern)".
9. **QA:** numbers must appear in the prompt, banned words, opening sentence and category.

### Changelog (v4.1)
- Dictionary v2.3: corrections from the historical validation (line_item_cnt grouping, qtyproportionzscore, round trips, data-linkage and policy wording); new Definition_Confidence and Level_Dependent columns.
- MIN_CONFIDENCE_FOR_MAIN setting; Main_Pattern_Confidence output column.
- Level-dependent measures name their analysis level in the prompt; a provider-level value is not quoted for a measure whose strongest signal came from another level.
- Encounter lines (ENC_CLM_IND) reported per category in the trust report.

### Changelog (v4.0)
- Input switched to the exploded explained_top_features file; ranking uses Feature_Rank_The_Provider and the 80% coverage tiers.
- Processes referred providers only; percentile selection with a floor; patient-mix status.
- Dictionary v2.2 (all 405 features of the 2026-10-12 run); count levels such as line_item_cnt_42 matched to `*_N` rows; missing-value levels (Unknown, Null, empty codes) never narrated.
- Peer comparisons from the 6-month and early scoring files, duplicate claim lines removed, only needed columns read.
- Lean QA (3 checks); scheme gating fixed (missing percentile or "Either" never confirms a scheme).
- Per-provider error handling, checkpoint and resume, optional batched generation.
- The mapping file used by v7 (`feature_mapping.csv`) is never read or written.

##### Dependencies
No `--force-reinstall`: it upgrades numpy and breaks older pandas. `kernels` lets transformers load GPT-OSS in MXFP4; without it the 120B model is expanded to bf16 (about 240 GB, more than one H100). If your current v7 install cell works on this cluster, you can keep it.

In [ ]:
%pip install transformers==5.2.0 accelerate sentencepiece openai-harmony kernels
dbutils.library.restartPython()

In [ ]:
import os
import re
import json
import math
import hashlib
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

from huggingface_hub import login
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

##### Configuration

In [ ]:
provider_id = 'Servicing_Provider_ID'
ROOT = 'sedo.senou@azahcccs.gov'
PROJECTS = f'/Workspace/Users/{ROOT}/Projects'
DATA_DIR = f'{PROJECTS}/all_projects_predictions/data'
EXTERNAL_DIR = f'/Workspace/Users/{ROOT}/data/external data'

with open(f'{DATA_DIR}/scoring_date.txt', 'r') as f:
    scoring_date = f.read().strip()

NOTEBOOK_VERSION = 'v4.1'
DICTIONARY_VERSION = 'v2.3'

# ── Inputs ──
IMPORTANCE_FILE = f'{DATA_DIR}/explained_top_features_{scoring_date}.csv'
# Saved by combine_all_projects_anomaly_predictions (one row per referred provider and category):
#   Servicing_Provider_ID, Category, Servicing_Provider_Anomaly_Score, *_Perspective_Score, Scoring_Date
REFERRAL_FILE = f'{DATA_DIR}/referred_providers_{scoring_date}.csv'
# If that file does not exist yet, the 8 referral CSVs are read and the same renaming and filter are applied.
REFERRAL_FALLBACK_TO_RAW = True
DICTIONARY_CSV = f'{DATA_DIR}/feature_mapping_v2_3.csv'        # never the v7 file feature_mapping.csv
SCORING_FILE_TYPES = ['rolling_6m_score', 'early_score']        # peer comparisons; duplicate lines removed

# Claim category -> (project folder, referral file prefix)
CATEGORY_FOLDERS = {
    'NURSING HOME': ('nursing_home', 'nh'),
    'NEMT NOT IHS': ('nemt_not_ihs', 'nemt_not_ihs'),
    'NEMT IHS': ('nemt_ihs', 'nemt_ihs'),
    'BH PROFESSIONAL': ('bh_professional_counseling', 'bh_lpc'),
    'BH OUTPATIENT CLINIC': ('bh_outpatient_clinic', 'bh_oc'),
    'BH RESIDENTIAL TREATMENT': ('bh_residential_treatment', 'bh_rt'),
    'DME AND APPLIANCES': ('dme_and_appliances', 'dme_and_appliances'),
    'DME MEDICAL SUPPLIES': ('dme_supplies', 'dme_supplies'),
}

# ── Outputs (same summary file names as v3.4; version in the Notebook_Version column) ──
OUTPUT_DIR = DATA_DIR
CHECKPOINT_FILE = f'{OUTPUT_DIR}/llm_summary_checkpoint_{scoring_date}_{NOTEBOOK_VERSION}.jsonl'
RESUME = True                       # skip providers already written to the checkpoint for this date and version

master_table_llm_summarization = "main.prod_results.master_anomaly_detection_referrals_and_llm_summarization"
test_table_llm_summarization = "main.sedo.test_anomaly_detection_referrals_and_llm_summarization"

# ── Which providers get an LLM narrative ──
NARRATE_PCTL = 0.90                 # per category, on Servicing_Provider_Anomaly_Score among referred providers
NARRATE_FLOOR = 5                   # at least this many narratives per category (if eligible)
NARRATE_CAP = None                  # optional maximum per category

# ── Patient mix ──
DEMOGRAPHIC_CATEGORIES = {'Patient Demographics'}      # Feature_Category values in the importance file
TOP_K_SIGNAL = 5                    # top-ranked features used to measure the demographic share
PATIENT_MIX_SHARE = 0.5             # demographic share of the top signal at or above which ...
PATIENT_MIX_TOP_DRIVERS = 3         # ... and no billing driver among this many top-ranked features

# ── Pattern selection ──
MAIN_TIERS = {'top_50', 'top_80'}   # Importance_Tier values inside the 80% signal coverage
MAX_MAIN = 3
MAX_SUPPORTING = 4
MAX_BACKGROUND = 3
STRICT_COVERAGE = True              # stop the run when a feature has no dictionary row
# Dictionary Definition_Confidence, weakest to strongest. A driver below MIN_CONFIDENCE_FOR_MAIN can still be a
# supporting pattern but never a main pattern. 'Unverified' allows every definition (most v2.2 rows are Unverified).
CONFIDENCE_ORDER = ['Unverified', 'Conceptually Supported', 'Historically Supported', 'Code Verified']
MIN_CONFIDENCE_FOR_MAIN = 'Unverified'
# Level_Dependent features: the same name is computed differently per analysis level, so the prompt states the level
LEVEL_PHRASE = {'provider': 'measured for the provider as a whole', 'patient': 'measured per patient, across the provider\'s patients',
                'visit': 'measured per visit', 'claim': 'measured per claim'}

# ── Python-computed values ──
MIN_PEERS = 10                      # providers needed in the category to quote a peer median and percentile
MIN_CLAIMS_FOR_VALUE = 20           # provider claims needed to quote its own value
HARMLESS_PCT_HIGH = 50              # a High-concern driver below this percentile becomes background
HARMLESS_PCT_LOW = 50               # a Low-concern driver above this percentile becomes background
SCHEME_PCT_HIGH = 75                # a scheme is confirmed for a High-concern pattern at or above this percentile
SCHEME_PCT_LOW = 25                 # ... and for a Low-concern pattern at or below this one

# ── Model ──
MODEL_ID = "openai/gpt-oss-120b"
REASONING_LEVEL = "low"
MAX_NEW_TOKENS = 800
RETRY_MAX_NEW_TOKENS = 1600
GEN_BATCH_SIZE = 1                  # >1 generates several notes per GPU call (left padding); test before raising
HF_SECRET_SCOPE = None              # Databricks secret scope holding the Hugging Face token (or set HF_TOKEN)
HF_SECRET_KEY = 'hf_token'

# ── Schemes: plain paid-claims wording; no intent, no medical-necessity judgment ──
SCHEME_DEFINITIONS_BY_LABEL = {
    'upcoding': 'payment for a more expensive code than peers are paid for with similar patients, or for a diagnosis rarely paired with that equipment or service among peers',
    'unbundling': 'payment for several line items or claims where peers are paid for one combined service',
    'modifier misuse': 'payment with modifiers that change reimbursement (for example RR = rental) in combinations peers rarely use',
    'diagnosis inflation/padding': 'unspecified or rarely seen diagnoses on paid claims far more often than at peers',
    'excessive visits': 'far more paid encounters per patient than peers have for the same type of service',
    'frequency abuse': 'paid units or services above a policy frequency or quantity limit, or far above peers for the same code',
    'phantom billing': 'paid services with no supporting activity in the claims data, such as trips with no medical claim for the member that day; this is a gap to check, not proof the service did not happen',
    'duplicate billing': 'the same service paid more than once for the same patient and date',
    'service upcreep': 'a shift toward higher-cost codes over time, faster than peers serving similar patients',
    'place-of-service manipulation': 'payment with a place of service that peers rarely pair with the procedure',
    'billing for non-covered services': "payment for services outside the member's covered benefit or eligibility, or for non-covered trip purposes; weekend or holiday dates are a timing pattern, not non-covered",
    'high-cost outlier manipulation': 'payments concentrated in a few high-cost codes or claims far above peers',
    'impossible day billing': 'time-based units for one patient on one day that add up to more hours than a day holds',
    'rental vs purchase manipulation': 'rental and purchase payments for the same equipment in patterns peers rarely show',
    'mileage inflation': 'paid transport miles or wait-time units far above what peers are paid for similar trips',
    'geographic improbability': 'trip distances or patient locations that are implausible for the service area',
    'level of care upcoding': 'payment at a higher level of care than peers use for similar patients',
    'group therapy upcoded to individual': 'individual therapy codes paid where peers mostly use group codes',
}
KNOWN_SCHEMES = [
    "upcoding", "unbundling", "excessive visits", "frequency abuse", "phantom billing",
    "duplicate billing", "service upcreep", "place-of-service manipulation",
    "modifier misuse", "medically unnecessary services",
    "billing for non-covered services", "high-cost outlier manipulation",
    "impossible day billing", "ineligible hospice enrollment",
    "medically unnecessary dme", "rental vs purchase manipulation",
    "mileage inflation", "geographic improbability",
    "level of care upcoding", "group therapy upcoded to individual",
    "diagnosis inflation/padding", "other",
]
NEW_PATTERN_LABEL = 'No known scheme (possible new pattern)'
# A single measure cannot support these labels (place of service alone is not a concern; trips or holidays alone do not
# show missing activity): they are confirmed only when at least two different main patterns link to them.
SCHEMES_NEEDING_TWO_MEASURES = {'place-of-service manipulation', 'phantom billing', 'duplicate billing',
                                'billing for non-covered services'}
CATEGORY_DISPLAY = {   # claim category inside Python-written value sentences
    'NEMT NOT IHS': 'NEMT (non-IHS)', 'NEMT IHS': 'NEMT (IHS)', 'DME MEDICAL SUPPLIES': 'DME medical supplies',
    'DME AND APPLIANCES': 'DME and appliances', 'NURSING HOME': 'nursing home',
    'BH OUTPATIENT CLINIC': 'BH outpatient clinic', 'BH RESIDENTIAL TREATMENT': 'BH residential treatment',
    'BH PROFESSIONAL': 'BH professional'}

CATEGORY_ALIASES = {
    'bh outpatient clinic': ['behavioral health outpatient clinic', 'behavioral health outpatient', 'bh outpatient'],
    'bh residential treatment': ['behavioral health residential treatment', 'behavioral health residential', 'bh residential'],
    'bh professional': ['behavioral health professional', 'bh professional counseling', 'behavioral health counseling'],
    'nemt not ihs': ['nemt (not ihs)', 'nemt (non-ihs)', 'non-ihs nemt', 'nemt non-ihs',
                     'non-emergency medical transportation (not ihs)', 'non-emergency medical transportation not ihs'],
    'nemt ihs': ['nemt (ihs)', 'ihs nemt', 'non-emergency medical transportation ihs'],
    'dme and appliances': ['durable medical equipment and appliances', 'dme appliances'],
    'dme medical supplies': ['durable medical equipment medical supplies', 'durable medical equipment supplies', 'dme supplies'],
    'nursing home': ['nursing facility', 'nursing home care'],
}
print(f"{NOTEBOOK_VERSION} | dictionary {DICTIONARY_VERSION} | scoring date {scoring_date}")

##### Helpers

In [ ]:
# ── Text and number helpers ──
# GPT-OSS writes narrow no-break spaces, non-breaking hyphens and curly quotes; every check runs on normalized text.
_UNI_MAP = str.maketrans({
    **{chr(c): '-' for c in (0x2010, 0x2011, 0x2012, 0x2013, 0x2014, 0x2015, 0x2212)},
    **{chr(c): ' ' for c in (0x00a0, 0x2007, 0x2009, 0x202f)},
    chr(0x200b): '', chr(0x2018): "'", chr(0x2019): "'", chr(0x201c): '"', chr(0x201d): '"', chr(0x00d7): 'x',
})
_DIGIT_GROUP_SPACE = re.compile('(?<=\\d)[' + chr(0x00a0) + chr(0x2007) + chr(0x2009) + chr(0x202f) + '](?=\\d{3}(?!\\d))')


def normalize_text(s):
    if s is None:
        return ''
    s = _DIGIT_GROUP_SPACE.sub(',', str(s))
    return unicodedata.normalize('NFKC', s).translate(_UNI_MAP)


def normalize_provider_id(series):
    """Provider IDs as clean strings ('1029', not 1029.0); blanks become NaN."""
    s = series.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return s.where(~s.isin(['', 'nan', 'None', 'NaN', '<NA>']))


def normalize_category(series):
    return series.astype(str).str.strip().str.upper()


def same_date(series, date_str):
    """True where a date column equals date_str, whatever its format ('2026-10-12', '2026-10-12 00:00:00', ...)."""
    want = pd.to_datetime(date_str).strftime('%Y-%m-%d')
    s = series.astype(str).str.strip()
    parsed = {v: pd.to_datetime(v, errors='coerce') for v in s.unique()}     # per value: mixed formats are fine
    return s.map(lambda v: (not pd.isna(parsed[v])) and parsed[v].strftime('%Y-%m-%d') == want).astype(bool)


def _norm_words(s):
    return ' ' + re.sub(r'[^a-z0-9]+', ' ', normalize_text(s).lower()).strip() + ' '


def category_mentioned(text, category):
    """True if the claim category (or an alias) appears as whole words; longer names of other categories are removed
    first so 'durable medical equipment supplies' does not count as DME AND APPLIANCES."""
    s = _norm_words(text)
    cat = str(category).lower().strip()
    own = [_norm_words(a) for a in [cat] + CATEGORY_ALIASES.get(cat, [])]
    for other, aliases in CATEGORY_ALIASES.items():
        if other == cat:
            continue
        for a in sorted([other] + aliases, key=len, reverse=True):
            na = _norm_words(a)
            if any(o.strip() in na.strip() for o in own) and na not in own:
                s = s.replace(na, ' ')
    if cat == 'nemt ihs' and re.search(r' no[nt] ihs ', s):
        return False
    return any(o in s for o in own)


def ordinal(n):
    n = int(round(float(n)))
    suf = 'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suf}"


def pct_ordinal(p):
    """Percentile as an ordinal between 1st and 99th."""
    return ordinal(min(max(float(p), 1), 99))


def percentile_of(values, v):
    """Percent of peers below v, ties counting half. values include v."""
    vals = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if vals.empty or v is None or pd.isna(v):
        return np.nan
    return float(((vals < v).mean() + 0.5 * (vals == v).mean()) * 100)


def fmt_value(x, vtype):
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return None
    if isinstance(x, str):
        return x
    x = float(x)
    if vtype == 'Dollars':
        return f"${x:,.0f}" if abs(x) >= 100 else f"${x:,.2f}"
    if vtype in ('Share (0-1)', 'Category level', 'Tier label', 'Yes/No flag'):
        if not 0 <= x <= 1:
            return f"{x:,.1f}"
        x = 0.999 if 0.999 <= x < 1 else (0.001 if 0 < x < 0.001 else x)
        return f"{x:.1%}"
    if vtype in ('Count', 'Days', 'Miles', 'Units'):
        return f"{x:,.0f}" if float(x).is_integer() else f"{x:,.1f}"
    return f"{x:,.2f}"

##### Code reference tables
Procedure, diagnosis and modifier descriptions for codes inside feature names.

In [ ]:
# ── Code reference tables (procedure, diagnosis, modifier descriptions) ──
icd10_desc = pd.read_csv(f'{EXTERNAL_DIR}/icd10_categories_description.csv')
hcpcs_desc = pd.read_csv(f'{EXTERNAL_DIR}/hcpcs_code_description.csv')
cpt_desc = pd.read_csv(f'{EXTERNAL_DIR}/cpt_codes_description.csv')
modifier_desc = pd.read_csv(f'{EXTERNAL_DIR}/procedure_modifiers.csv')
diag_hierarchy = pd.read_csv(f'{EXTERNAL_DIR}/icd_10_diagnosis_code_root_and_category.csv', dtype=str)


def _build_lookup(df, candidate_pairs):
    for code_col, desc_col in candidate_pairs:
        if code_col in df.columns and desc_col in df.columns:
            d = df.dropna(subset=[code_col, desc_col])
            return dict(zip(d[code_col].astype(str).str.strip().str.upper(), d[desc_col].astype(str).str.strip()))
    cols = df.columns.tolist()
    if len(cols) >= 2:
        d = df.dropna(subset=cols[:2])
        return dict(zip(d[cols[0]].astype(str).str.strip().str.upper(), d[cols[1]].astype(str).str.strip()))
    return {}


proc_code_lookup = _build_lookup(hcpcs_desc, [('HCPCS Codes', 'Description'), ('hcpcs_code', 'description'),
                                              ('Code', 'Description'), ('HCPCS_CD', 'Description')])
for k, v in _build_lookup(cpt_desc, [('CPT Code', 'Description'), ('cpt_code', 'description'),
                                     ('Code', 'Description'), ('CPT_CD', 'Description')]).items():
    proc_code_lookup.setdefault(k, v)
diag_code_lookup = {k.replace('.', ''): v for k, v in _build_lookup(icd10_desc, [
    ('ICD10 Code', 'Description'), ('icd10_code', 'description'), ('Code', 'Description'),
    ('DIAG_CD', 'Description'), ('ICD-10-CM Code', 'Description')]).items()}
modifier_lookup = _build_lookup(modifier_desc, [('Modifier', 'Description'), ('modifier', 'description'),
                                                ('Code', 'Description'), ('PROC_MOD', 'Description'),
                                                ('Modifier Code', 'Modifier Description')])
_dh = diag_hierarchy.rename(columns=lambda c: c.strip()).rename(columns={'diag_cd_root': 'diag_root'})
_dh['diag_root'] = _dh['diag_root'].astype(str).str.strip().str.upper()
if _dh['diag_root'].duplicated().any():
    print(f"WARNING: {int(_dh['diag_root'].duplicated().sum())} duplicate roots in the ICD-10 group file (first kept).")
_dh = _dh.drop_duplicates('diag_root')
diag_group_lookup = {k: v for k, v in zip(_dh['diag_root'], _dh['category_level_III'].fillna('').astype(str).str.strip()) if v}
print(f"Codes described: procedures {len(proc_code_lookup):,} | diagnoses {len(diag_code_lookup):,} | "
      f"diagnosis groups {len(diag_group_lookup):,} | modifiers {len(modifier_lookup):,}")

CODES_WITHOUT_DESCRIPTION = Counter()
CODES_DESCRIBED_BY_GROUP = Counter()


def _cap(s):
    s = str(s).strip()
    return s[:1].upper() + s[1:] if s else s


def describe_diag(code):
    c = str(code).strip().upper().replace('.', '')
    if not c:
        return ''
    if c in diag_code_lookup:
        return f"Diagnosis {c}: {_cap(diag_code_lookup[c])}"
    if c[:3] in diag_group_lookup:
        CODES_DESCRIBED_BY_GROUP[c] += 1
        return f"Diagnosis {c} (group {c[:3]}: {_cap(diag_group_lookup[c[:3]])})"
    CODES_WITHOUT_DESCRIPTION[c] += 1
    return f"Diagnosis {c}"


def describe_proc(code):
    c = str(code).strip().upper()
    if c in ('', '0', '-1'):
        return ''
    if c in proc_code_lookup:
        return f"Procedure {c}: {_cap(proc_code_lookup[c])}"
    CODES_WITHOUT_DESCRIPTION[c] += 1
    return f"Procedure {c}"


def describe_mods(mod_part):
    out = []
    for m in [m.strip().upper() for m in str(mod_part).split(',') if m.strip() not in ('-1', '0', '')]:
        if m in modifier_lookup:
            out.append(f"{m}={_cap(modifier_lookup[m])}")
        else:
            CODES_WITHOUT_DESCRIPTION[m] += 1
            out.append(m)
    return f"Modifiers: {', '.join(out)}" if out else ''


def decode_feature_codes(feature):
    """Codes embedded in a feature name, with their descriptions: 'Procedure E1390: ... | Modifiers: RR=Rental'."""
    s = str(feature)
    parts = []
    for prefix in ('procmodif_proc_pair_', 'proc_proc_mod_other_'):
        if s.startswith(prefix) and '__' in s:
            mods, proc = s[len(prefix):].split('__', 1)
            parts = [describe_proc(proc), describe_mods(mods)]
            break
    else:
        m = re.match(r'^(?:diag_cd_\d_proc_cd_pair_|diag_proc_other_)(.+?)__(.+)$', s)
        if m:
            parts = [describe_diag(m.group(1)), describe_proc(m.group(2))]
        elif s.startswith('PROC_CD_'):
            parts = [describe_proc(s[len('PROC_CD_'):])]
        elif re.match(r'^PROC_MOD_\d_', s):
            parts = [describe_mods(s.split('_', 3)[3])]
    return ' | '.join(p for p in parts if p)

##### Feature dictionary (v2.2)
One row per model feature root; one-hot levels (`POS_DESC_HOME`) and count levels (`line_item_cnt_42` -> `line_item_cnt_N`) map to their root.

In [ ]:
# ── Feature dictionary v2.2 ──
mapping_df = pd.read_csv(DICTIONARY_CSV, dtype=str).fillna('')
REQUIRED_MAPPING_COLS = ['Feature_Root', 'Replacement', 'Description', 'Suspicious_Pattern', 'Benign_Explanations',
                         'Narrative_Caveat', 'Value_Template', 'Records_To_Review', 'Policy_Lever', 'Narrate_Role',
                         'Validation_Status', 'Applies_To', 'Concept_Group', 'Measured_At', 'Value_Type',
                         'Scheme_Link', 'Concern_Direction']
_missing_cols = [c for c in REQUIRED_MAPPING_COLS if c not in mapping_df.columns]
assert not _missing_cols, f"{DICTIONARY_CSV} is not a v2.2 export; missing columns: {_missing_cols}"
for c in ('Definition_Confidence', 'Level_Dependent'):          # v2.3 columns; older exports lack them
    if c not in mapping_df.columns:
        mapping_df[c] = 'Unverified' if c == 'Definition_Confidence' else ''
for c in mapping_df.columns:
    mapping_df[c] = mapping_df[c].astype(str).str.strip()
mapping_df.loc[~mapping_df['Definition_Confidence'].isin(CONFIDENCE_ORDER), 'Definition_Confidence'] = 'Unverified'
assert mapping_df['Feature_Root'].is_unique, "Duplicate Feature_Root rows in the dictionary"
assert (mapping_df['Description'] != '').all(), "Every dictionary row must have a Description"
assert set(mapping_df['Narrate_Role']) <= {'Driver', 'Context only', 'Do not narrate'}, set(mapping_df['Narrate_Role'])
assert set(mapping_df['Concern_Direction']) <= {'High', 'Low', 'Either', ''}, set(mapping_df['Concern_Direction'])
_bad_links = {s.strip() for v in mapping_df['Scheme_Link'] for s in v.split('|')
              if s.strip() and s.strip().lower() != 'none' and s.strip().lower() not in KNOWN_SCHEMES}
assert not _bad_links, f"Scheme_Link labels not in KNOWN_SCHEMES: {_bad_links}"

MAP = mapping_df.set_index('Feature_Root').to_dict('index')
ONE_HOT_VALUE_TYPES = {'Category level', 'Tier label', 'Yes/No flag', 'Date part'}
SUFFIX_OK_ROOTS = sorted((r for r, v in MAP.items() if v['Value_Type'] in ONE_HOT_VALUE_TYPES), key=len, reverse=True)
COUNT_LEVEL_ROOTS = {r for r, v in MAP.items() if v['Value_Type'] == 'Count level'}     # e.g. line_item_cnt_N


def match_root(feature):
    """Dictionary row of a model feature: exact name; one-hot root + '_level'; or count level 'name_42' -> 'name_N'."""
    s = str(feature).strip()
    if s in MAP:
        return s
    for r in SUFFIX_OK_ROOTS:
        if s.startswith(r + '_'):
            return r
    m = re.match(r'^(.+)_(\d+)$', s)
    if m and f'{m.group(1)}_N' in COUNT_LEVEL_ROOTS:
        return f'{m.group(1)}_N'
    return ''


# ── Levels carried in one-hot feature names, in plain words ──
TIER_ROOTS = {'average_procedure_zscore_class', 'global_procedure_zscore_class', 'mileage_zscore_class',
              'not_mileage_zscore_class', 'procedure_volume_zscore_class', 'reservation_zscore_class',
              'wait_time_zscore_class'}
TIER_PLAIN = {   # z-scores are capped at 10 upstream, so 'Rare Anomaly' is the highest tier reached
    'extreme low underuse': 'far below peers', 'very low underuse': 'well below peers',
    'slightly below average underuse': 'below peers', 'slightly below avg underuse': 'below peers',
    'normal': 'in line with peers, or too little data to compare',
    'mild anomaly': 'somewhat above peers', 'anomaly': 'well above peers',
    'rare anomaly': 'far above peers (highest level)', 'extreme anomaly': 'far above peers (highest level)',
    'insanely rare': 'far above peers (highest level)',
}
WAIT_BANDS = {   # unusual_wait_time_class: fixed hours (1 T2007 unit = 30 minutes)
    'normal': 'up to 2 hours', 'high': '2 to 4 hours', 'extremely high': '4 to 8 hours',
    'abnormal': '8 to 24 hours', 'suspicious': 'more than 24 hours', 'no wait time': 'no wait time',
}
MULTI_DIM_PLAIN = {
    'normal': 'no measure unusual', 'single dimension': 'unusual on one measure',
    'multi dimension moderate': 'unusual on several measures', 'multi dimension high': 'unusual on many measures',
    'multi dimension extreme': 'unusual on most measures',
}
# Levels saying the provider is normal or below peers, or a flag is absent: background only, never a reason
NON_CONCERNING_LEVELS = {'in line with peers, or too little data to compare', 'below peers', 'well below peers',
                         'far below peers', 'not present', 'up to 2 hours', 'no wait time',
                         'below the top 5% of providers', 'no measure unusual'}
# Levels that only say a value was missing: never narrated
MISSING_LEVEL_WORDS = {'', 'unknown', 'null', 'nan', 'none', 'missing', 'other unknown', 'not available', 'n/a'}
RISKY_LEVEL_WORDS = re.compile(r'anomal|suspicious|rare|outlier|extreme|insane|abnormal|watch', re.I)
UNKNOWN_LEVELS = set()


def raw_level(feature, root):
    s = str(feature).strip()
    if root in COUNT_LEVEL_ROOTS:
        return s.rsplit('_', 1)[1]
    return s[len(root) + 1:] if len(s) > len(root) else ''


def _clean_code_level(raw):
    """'RR,N1,-1,-1__E1390' -> 'RR, N1 with E1390' (placeholders -1 and 0 removed); '' when only placeholders."""
    out = []
    for part in raw.split('__'):
        toks = [t.strip() for t in part.split(',') if t.strip() not in ('', '-1', '0')]
        if toks:
            out.append(', '.join(toks))
    return ' with '.join(out)


def plain_level(feature, root):
    """(level text, is_missing) for a one-hot or count-level feature; ('', False) for a plain feature."""
    raw = raw_level(feature, root)
    if not raw:
        return '', False
    key = raw.replace('_', ' ').strip().lower()
    vtype = MAP[root]['Value_Type']
    if root in COUNT_LEVEL_ROOTS:
        return raw, False
    if root in TIER_ROOTS:
        if key in TIER_PLAIN:
            return TIER_PLAIN[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    if root == 'unusual_wait_time_class':
        if key in WAIT_BANDS:
            return WAIT_BANDS[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual wait-time band', False
    if root == 'multi_dim_anomaly_class':
        if key in MULTI_DIM_PLAIN:
            return MULTI_DIM_PLAIN[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    if root in ('visit_frequency_class', 'weeks_with_unusual_visits_class'):
        m = re.search(r'above\s*(\d+)', key)
        if m:
            return f"top {100 - int(m.group(1))}% of providers", False
        if 'below' in key:
            return 'below the top 5% of providers', False
    if vtype == 'Yes/No flag':
        if key in ('1', '1.0', 'true', 'y', 'yes'):
            return '', False
        if key in ('0', '0.0', 'false', 'n', 'no'):
            return 'not present', False
    if vtype == 'Category level' and ('__' in raw or ',' in raw):
        cleaned = _clean_code_level(raw)
        return cleaned, cleaned == ''
    if key in MISSING_LEVEL_WORDS:
        return raw, True
    if vtype == 'Tier label' and RISKY_LEVEL_WORDS.search(key):
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    return raw.replace('_', ' ').strip(), False


def feature_label(root, level):
    label = MAP[root]['Replacement']
    if '{level}' in label:
        return label.replace('{level}', level)
    return f"{label}: {level}" if level else label


def applies_to(applies, category):
    a, cat = str(applies).strip().upper(), str(category).strip().upper()
    if a in ('', 'ALL CATEGORIES'):
        return True
    if a.startswith('ALL EXCEPT'):
        return cat not in [x.strip() for x in a.replace('ALL EXCEPT', '').split(';')]
    return cat in [x.strip() for x in a.split(';')]


print(f"Dictionary {DICTIONARY_VERSION}: {len(MAP)} rows | Driver {sum(v['Narrate_Role'] == 'Driver' for v in MAP.values())} | "
      f"Context only {sum(v['Narrate_Role'] == 'Context only' for v in MAP.values())} | on hold "
      f"{sum(v['Validation_Status'] == 'Hold until confirmed' for v in MAP.values())} | one-hot roots {len(SUFFIX_OK_ROOTS)} | "
      f"count-level rows {len(COUNT_LEVEL_ROOTS)}")

##### Load the exploded importance file

In [ ]:
# ── Exploded importance file: one row per provider x Analysis_Level x feature ──
IMPORTANCE_COLS = [provider_id, 'Category', 'Analysis_Level', 'Importance_Tier', 'Analysis_Levels_In_Agreement',
                   'Which_Analysis_Levels', 'Feature_Name', 'Feature_Category', 'Feature_Rank_The_Provider',
                   'Overall_Importance_Score', 'Share_Of_Provider_Signal', 'Cumulative_Signal_Coverage', 'Scoring_Date']
_imp_raw = pd.read_csv(IMPORTANCE_FILE, dtype={provider_id: str}, low_memory=False)
_missing = [c for c in IMPORTANCE_COLS if c not in _imp_raw.columns]
if _missing:
    raise ValueError(f"{IMPORTANCE_FILE} is not the exploded explained_top_features file; missing {_missing}")
imp = _imp_raw[IMPORTANCE_COLS].copy()
imp[provider_id] = normalize_provider_id(imp[provider_id])
imp['Category'] = normalize_category(imp['Category'])
imp['Feature'] = imp['Feature_Name'].astype(str).str.strip()
imp['Feature_Category'] = imp['Feature_Category'].fillna('').astype(str).str.strip()
imp['Importance_Tier'] = imp['Importance_Tier'].fillna('').astype(str).str.strip()
for c in ['Feature_Rank_The_Provider', 'Overall_Importance_Score', 'Analysis_Levels_In_Agreement']:
    imp[c] = pd.to_numeric(imp[c], errors='coerce')

load_stats = Counter(importance_rows=len(imp))
_date_ok = same_date(imp['Scoring_Date'], scoring_date)
load_stats['dropped_other_scoring_date'] = int((~_date_ok).sum())
imp = imp[_date_ok]
if imp.empty:
    raise ValueError(f"No rows for scoring date {scoring_date} in {IMPORTANCE_FILE}; dates in file: "
                     f"{sorted(_imp_raw['Scoring_Date'].astype(str).unique())[:5]}")
_bad = imp[provider_id].isna() | imp['Overall_Importance_Score'].isna() | imp['Feature_Rank_The_Provider'].isna()
load_stats['dropped_blank_id_score_or_rank'] = int(_bad.sum())
imp = imp[~_bad]
_idx = imp['Feature'].str.lower() == 'index'
load_stats['dropped_index_feature'] = int(_idx.sum())
imp = imp[~_idx]

_unknown_tiers = sorted(set(imp['Importance_Tier']) - MAIN_TIERS)
print(f"Importance rows: {len(imp):,} | providers {imp[provider_id].nunique()} | "
      f"levels {imp['Analysis_Level'].value_counts().to_dict()}")
print(f"Importance_Tier values: {imp['Importance_Tier'].value_counts().to_dict()} "
      f"(main patterns come only from {sorted(MAIN_TIERS)})")
print("Load:", dict(load_stats))

##### Load referred providers

In [ ]:
# ── Referred providers and their anomaly score ──
REFERRAL_COLS = [provider_id, 'Category', 'Servicing_Provider_Anomaly_Score', 'Provider_Perspective_Score',
                 'Patient_Perspective_Score', 'Visit_Perspective_Score', 'Claim_Perspective_Score']

if os.path.exists(REFERRAL_FILE):
    referrals = pd.read_csv(REFERRAL_FILE, dtype={provider_id: str})
    REFERRAL_SOURCE = REFERRAL_FILE
    if 'Scoring_Date' in referrals.columns:
        _ok = same_date(referrals['Scoring_Date'], scoring_date)
        if not _ok.all():
            print(f"WARNING: dropping {int((~_ok).sum())} referral rows with another Scoring_Date")
        referrals = referrals[_ok]
elif REFERRAL_FALLBACK_TO_RAW:
    # Same steps as combine_all_projects_anomaly_predictions (rename, category names, BH PROFESSIONAL payout filter).
    # Keep in sync, or save referred_providers_{scoring_date}.csv from that notebook instead.
    print(f"WARNING: {REFERRAL_FILE} not found; reading the 8 referral files directly.")
    _raw_cat = {prefix: cat for cat, (_, prefix) in CATEGORY_FOLDERS.items()}
    _parts = []
    for cat, (folder, prefix) in CATEGORY_FOLDERS.items():
        p = f'{PROJECTS}/{folder}/referrals/{prefix}_monthly_anomaly_detection_referrals.csv'
        if os.path.exists(p):
            _parts.append(pd.read_csv(p, dtype={'SProv_ID': str}))
        else:
            print(f"  missing referral file for {cat}: {p}")
    referrals = pd.concat(_parts, ignore_index=True)
    referrals['category'] = referrals['category'].replace(_raw_cat)
    referrals = referrals.rename(columns={
        'category': 'Category', 'SProv_ID': provider_id, 'monthly_payout': 'Average_Monthly_Payout',
        'final_weighted_provider_score': 'Servicing_Provider_Anomaly_Score', 'score_provider': 'Provider_Perspective_Score',
        'score_patient': 'Patient_Perspective_Score', 'score_visit': 'Visit_Perspective_Score',
        'score_claim': 'Claim_Perspective_Score'})
    referrals['Category'] = normalize_category(referrals['Category'])
    referrals = referrals[~((referrals['Category'] == 'BH PROFESSIONAL')
                            & (pd.to_numeric(referrals['Average_Monthly_Payout'], errors='coerce') < 20000))]
    REFERRAL_SOURCE = 'raw referral files'
else:
    raise FileNotFoundError(REFERRAL_FILE)

for c in REFERRAL_COLS:
    if c not in referrals.columns:
        if c == 'Servicing_Provider_Anomaly_Score' or c in (provider_id, 'Category'):
            raise ValueError(f"Referral data has no {c}; columns: {list(referrals.columns)}")
        referrals[c] = np.nan
referrals = referrals[REFERRAL_COLS].copy()
referrals[provider_id] = normalize_provider_id(referrals[provider_id])
referrals['Category'] = normalize_category(referrals['Category'])
for c in REFERRAL_COLS[2:]:
    referrals[c] = pd.to_numeric(referrals[c], errors='coerce')
referrals = referrals.dropna(subset=[provider_id])
_dups = int(referrals.duplicated([provider_id, 'Category']).sum())
if _dups:
    print(f"NOTE: {_dups} repeated provider-category referral rows; keeping the highest score.")
referrals = (referrals.sort_values('Servicing_Provider_Anomaly_Score', ascending=False)
             .drop_duplicates([provider_id, 'Category']).reset_index(drop=True))

# Process referred providers only
_ref_keys = set(zip(referrals[provider_id], referrals['Category']))
_imp_keys = set(zip(imp[provider_id], imp['Category']))
load_stats['referred_provider_categories'] = len(_ref_keys)
load_stats['importance_only_not_referred'] = len(_imp_keys - _ref_keys)
load_stats['referred_without_feature_rows'] = len(_ref_keys - _imp_keys)
imp = imp[[k in _ref_keys for k in zip(imp[provider_id], imp['Category'])]].copy()
print(f"Referral source: {REFERRAL_SOURCE}")
print(f"Referred provider-categories: {len(_ref_keys)} | with feature rows: {len(_ref_keys & _imp_keys)} | "
      f"in importance file but not referred (skipped): {len(_imp_keys - _ref_keys)} | "
      f"referred with no feature rows: {len(_ref_keys - _imp_keys)}")
print(referrals['Category'].value_counts().to_string())

##### Load scoring data (peer comparisons)

In [ ]:
# ── Scoring files (all providers in each category): peer comparisons and provider values ──
# Only the columns needed are read: claim keys and amounts, plus any column named like a dictionary root.
BASE_SCORING_COLS = ['SProv_ID', provider_id, 'UNIQUE_KEY', 'claim_id', 'MEMBER_KEY', 'PMT_AMT', 'QUANTITY_PAID',
                     'PROC_CD', 'proc_cd_category', 'diag_1_level_I', 'ENC_CLM_IND']
_needed_roots = {match_root(f) for f in imp['Feature'].unique()} - {''}
_wanted = set(BASE_SCORING_COLS) | {r for r in _needed_roots if r not in COUNT_LEVEL_ROOTS}
_frames, scoring_stats = [], Counter()
for cat, (folder, _) in CATEGORY_FOLDERS.items():
    for stype in SCORING_FILE_TYPES:
        p = f'{PROJECTS}/{folder}/scoring_data/{stype}_monthly_scoring_file.csv.gz'
        if not os.path.exists(p):
            print(f"  missing scoring file: {p}")
            continue
        header = pd.read_csv(p, compression='gzip', nrows=0).columns
        cols = [c for c in header if c in _wanted]
        part = pd.read_csv(p, compression='gzip', usecols=cols, low_memory=False)
        part['Category'] = cat
        part['_file'] = stype
        _frames.append(part)
        scoring_stats[f'lines_{stype}'] += len(part)
df_scoring_master = pd.concat(_frames, ignore_index=True)
del _frames
if 'SProv_ID' in df_scoring_master.columns:
    if provider_id in df_scoring_master.columns:
        df_scoring_master[provider_id] = df_scoring_master[provider_id].fillna(df_scoring_master['SProv_ID'])
        df_scoring_master = df_scoring_master.drop(columns=['SProv_ID'])
    else:
        df_scoring_master = df_scoring_master.rename(columns={'SProv_ID': provider_id})
df_scoring_master[provider_id] = normalize_provider_id(df_scoring_master[provider_id])
df_scoring_master = df_scoring_master.dropna(subset=[provider_id])
# The 6-month and early files can hold the same claim line; keep it once (6-month file first)
if 'UNIQUE_KEY' in df_scoring_master.columns:
    _n = len(df_scoring_master)
    df_scoring_master = df_scoring_master.drop_duplicates(['Category', 'UNIQUE_KEY'], keep='first')
    scoring_stats['duplicate_lines_removed'] = _n - len(df_scoring_master)
else:
    print("WARNING: no UNIQUE_KEY column; claim lines present in both scoring files are counted twice.")
if 'claim_id' not in df_scoring_master.columns:
    df_scoring_master['claim_id'] = df_scoring_master.get('UNIQUE_KEY', pd.Series(range(len(df_scoring_master))))
df_scoring_master = df_scoring_master.drop(columns=['_file']).reset_index(drop=True)
_keys = {provider_id, 'UNIQUE_KEY', 'claim_id', 'MEMBER_KEY', 'Category'}
for c in df_scoring_master.columns:          # text columns as category: much less memory
    if c not in _keys and df_scoring_master[c].dtype == object:
        df_scoring_master[c] = df_scoring_master[c].astype('category')
print(f"Scoring lines: {len(df_scoring_master):,} | columns read: {df_scoring_master.shape[1]} | {dict(scoring_stats)}")
print(df_scoring_master.groupby('Category')[provider_id].nunique().rename('providers').to_string())

##### Peer comparison

In [ ]:
# ── Provider size and payment figures vs all providers in the same claim category (computed once) ──
PROFILE_METRICS = [   # (column, label, format)
    ('total_paid', 'Total paid', 'Dollars'),
    ('total_claims', 'Paid claims', 'Count'),
    ('unique_patients', 'Patients with paid claims', 'Count'),
    ('avg_paid_per_claim', 'Average paid per claim', 'Dollars'),
    ('claims_per_patient', 'Paid claims per patient', 'Ratio'),
    ('lines_per_claim', 'Paid lines per claim', 'Ratio'),
]
cohort_stats = {}
for cat, grp in df_scoring_master.groupby('Category'):
    g = grp.groupby(provider_id)
    st = pd.DataFrame({
        'total_paid': g['PMT_AMT'].sum() if 'PMT_AMT' in grp else np.nan,
        'total_claims': g['claim_id'].nunique(),
        'unique_patients': g['MEMBER_KEY'].nunique() if 'MEMBER_KEY' in grp else np.nan,
        'total_lines': g.size(),
    })
    st['avg_paid_per_claim'] = st['total_paid'] / st['total_claims'].replace(0, np.nan)
    st['claims_per_patient'] = st['total_claims'] / st['unique_patients'].replace(0, np.nan)
    st['lines_per_claim'] = st['total_lines'] / st['total_claims'].replace(0, np.nan)
    cohort_stats[cat] = st
print("Peer cohorts:", {c: len(s) for c, s in cohort_stats.items()})


def peer_comparison_lines(pid, cat):
    """Provider's own figures with the peer median and percentile; [] when the provider has no scoring lines."""
    st = cohort_stats.get(cat)
    if st is None or pid not in st.index or len(st) < MIN_PEERS:
        return []
    out = [f"Compared with {len(st)} {cat} providers (paid claims, same period):"]
    for col, label, vtype in PROFILE_METRICS:
        v = st.at[pid, col]
        if pd.isna(v):
            continue
        out.append(f"  {label}: {fmt_value(v, vtype)} for this provider; peer median "
                   f"{fmt_value(st[col].median(), vtype)}; {pct_ordinal(percentile_of(st[col], v))} percentile")
    return out


def main_services(pid, cat, n=3):
    """Top procedure categories by share of the provider's paid lines."""
    if 'proc_cd_category' not in df_scoring_master.columns:
        return ''
    lines = df_scoring_master.loc[(df_scoring_master['Category'] == cat) & (df_scoring_master[provider_id] == pid),
                                  'proc_cd_category'].dropna()
    if lines.empty:
        return ''
    vc = lines.value_counts(normalize=True).head(n)
    return ', '.join(f"{k} ({v:.0%} of paid lines)" for k, v in vc.items())

##### Provider values (Python-computed)

In [ ]:
# ── Provider values for Value_Template sentences (Python only; the LLM copies them) ──
# Filled only when the scoring files hold the measure unambiguously:
#   A. one-hot level of a claim-line column (code, category, tier, yes/no): share of the provider's paid lines
#   B. a numeric column constant within the provider and within almost every provider (a provider-level measure)
# Peers = all providers in the same claim category with the value. Standardized scores are never quoted.
NON_NEGATIVE_TYPES = {'Count', 'Days', 'Miles', 'Units', 'Dollars'}
UNIT_INTERVAL_TYPES = {'Share (0-1)', 'Index (0-1)'}
NEVER_QUOTE_TYPES = {'Standardized score', 'Date part', 'Count level'}
_YES = {'1', 'y', 'yes', 'true', 't'}
_NO = {'0', 'n', 'no', 'false', 'f'}


def _norm_one(x):
    return re.sub(r'^(-?\d+)\.0+$', r'\1', str(x).strip().lower()).replace(' ', '_')


def _norm_level_series(col):
    """Normalized level labels; for category columns only the (few) categories are normalized."""
    if isinstance(col.dtype, pd.CategoricalDtype):
        return col.map({c: _norm_one(c) for c in col.cat.categories})
    return col.astype(str).map(_norm_one)


def _flag_series(col):
    s = col.astype(str).str.strip().str.lower().str.replace(r'^(-?\d+)\.0+$', r'\1', regex=True)
    return s.map(lambda x: 1.0 if x in _YES else (0.0 if x in _NO else np.nan))


def _looks_scaled(vtype, per_provider):
    v = per_provider.dropna()
    if v.empty:
        return True
    if vtype in UNIT_INTERVAL_TYPES:
        return bool(((v < 0) | (v > 1)).any())
    if vtype in NON_NEGATIVE_TYPES:
        return bool((v < 0).any() or (v.between(0, 1).all() and v.nunique() > 2))
    return False


def build_provider_feature_values(needed):
    """needed: DataFrame [provider_id, 'Category', 'Feature', 'Feature_Root'].
    Returns (values_df [provider_id, Category, Feature, value, peer_median, peer_pct, n_claims, value_source], coverage)."""
    rows, coverage = [], Counter()
    for cat, need_cat in needed.groupby('Category'):
        grp = df_scoring_master[df_scoring_master['Category'] == cat]
        if grp.empty:
            coverage['category_not_in_scoring_data'] += need_cat['Feature'].nunique()
            continue
        pids = grp[provider_id]
        n_claims = grp.groupby(provider_id)['claim_id'].nunique()
        norm_cache = {}
        for (feat, root), need_f in need_cat.groupby(['Feature', 'Feature_Root']):
            meta = MAP[root]
            vtype, measured_at = meta['Value_Type'], meta['Measured_At']
            if vtype in NEVER_QUOTE_TYPES or not meta['Value_Template']:
                coverage['not_quoted_by_design'] += 1
                continue
            if root not in grp.columns:
                coverage['column_not_in_scoring_data'] += 1
                continue
            col = grp[root]
            if vtype in ONE_HOT_VALUE_TYPES:
                if measured_at != 'Claim line':
                    coverage['share_of_lines_would_mislabel'] += 1
                    continue
                if vtype == 'Yes/No flag':      # the template describes the 'yes' condition, also for a '_0' feature
                    ind = _flag_series(col) == 1
                else:
                    if root not in norm_cache:
                        norm_cache[root] = _norm_level_series(col)
                    ind = norm_cache[root] == _norm_one(raw_level(feat, root))
                if not ind.any():
                    coverage['level_not_found_in_column'] += 1
                    continue
                per_provider = ind.groupby(pids).mean()
                source = 'share_of_lines'
            else:
                num = pd.to_numeric(col, errors='coerce')
                if num.notna().sum() == 0:
                    num = _flag_series(col)
                spread = num.groupby(pids).nunique(dropna=True)
                if (spread > 1).mean() > 0.01:
                    coverage['varies_within_provider'] += 1
                    continue
                per_provider = num.groupby(pids).first().dropna()
                if _looks_scaled(vtype, per_provider):
                    coverage['out_of_range_looks_scaled'] += 1
                    continue
                source = 'provider_measure'
            if len(per_provider) < MIN_PEERS:
                coverage['too_few_peers'] += 1
                continue
            peer_median = float(per_provider.median())
            for pid in need_f[provider_id].unique():
                if pid not in per_provider.index:
                    coverage['provider_not_in_scoring_data'] += 1
                    continue
                if source == 'provider_measure' and spread.get(pid, 0) > 1:
                    coverage['varies_within_this_provider'] += 1      # this provider's own lines disagree
                    continue
                v = float(per_provider.at[pid])
                rows.append({provider_id: pid, 'Category': cat, 'Feature': feat, 'value': v,
                             'peer_median': peer_median, 'peer_pct': percentile_of(per_provider, v),
                             'n_claims': int(n_claims.get(pid, 0)), 'value_source': source})
                coverage[source] += 1
    cols = [provider_id, 'Category', 'Feature', 'value', 'peer_median', 'peer_pct', 'n_claims', 'value_source']
    return pd.DataFrame(rows, columns=cols), coverage


def render_value_sentence(row):
    """Fill the dictionary Value_Template; '' unless every placeholder can be filled honestly."""
    tpl = MAP[row['Feature_Root']]['Value_Template']
    if not tpl or pd.isna(row.get('value')):
        return ''
    if row.get('n_claims', 0) < MIN_CLAIMS_FOR_VALUE:
        return ''
    vtype = MAP[row['Feature_Root']]['Value_Type']
    vals = {'value': fmt_value(row['value'], vtype), 'peer_median': fmt_value(row['peer_median'], vtype),
            'peer_pct': pct_ordinal(row['peer_pct']) if not pd.isna(row['peer_pct']) else None,
            'level': row.get('Level_Plain') or None, 'category': CATEGORY_DISPLAY.get(row['Category'], row['Category'])}
    needed = set(re.findall(r'\{(\w+)\}', tpl))
    if any(vals.get(k) in (None, '') for k in needed):
        return ''
    out = tpl.format(**{k: vals[k] for k in needed})
    if vtype == 'Units' and 'wait' in row['Feature'].lower() and 'unit' in row['Feature'].lower():
        hrs = float(row['value']) * 0.5                 # T2007: 1 unit = 30 minutes
        out = out.replace(f"{vals['value']} ", f"{vals['value']} (about {hrs:,.1f} hours) ", 1)
    return out

##### Prepare features
Coverage gate, roles, values, main / supporting / background patterns, patient mix, one row per referred provider.

In [ ]:
# ── Prepare features for every referred provider ──
prep_stats = Counter()
_k = [provider_id, 'Category', 'Feature']

# (1) One row per feature: the strongest analysis level is kept, with the levels it appeared at
imp = imp.sort_values(_k + ['Feature_Rank_The_Provider', 'Overall_Importance_Score'],
                      ascending=[True, True, True, True, False], kind='stable')
_lv = imp.groupby(_k).agg(Levels_In_File=('Analysis_Level', lambda s: ','.join(sorted(set(map(str, s))))),
                          In_Main_Tier=('Importance_Tier', lambda s: bool(set(s) & MAIN_TIERS))).reset_index()
feat = imp.drop_duplicates(_k, keep='first').merge(_lv, on=_k, how='left', validate='1:1')
feat = feat.rename(columns={'Analysis_Level': 'Best_Level'})
prep_stats['feature_rows_all_levels'] = len(imp)
prep_stats['feature_rows'] = len(feat)

# (2) Coverage gate: every feature needs a dictionary row
_roots = {f: match_root(f) for f in feat['Feature'].unique()}
feat['Feature_Root'] = feat['Feature'].map(_roots)
_unmapped = sorted(f for f, r in _roots.items() if not r)
if _unmapped:
    _path = f'{OUTPUT_DIR}/llm_summary_unmapped_features_{scoring_date}.csv'
    (feat[feat['Feature_Root'] == ''][['Feature', 'Feature_Category']].drop_duplicates()
        .to_csv(_path, index=False))
    msg = (f"{len(_unmapped)} features have no dictionary row (list saved to {_path}); add them to the dictionary:\n  "
           + "\n  ".join(_unmapped[:30]))
    if STRICT_COVERAGE:
        raise ValueError(msg)
    print("WARNING:", msg)
    prep_stats['dropped_unmapped'] = int((feat['Feature_Root'] == '').sum())
    feat = feat[feat['Feature_Root'] != '']

# (3) Dictionary fields, plain labels, decoded codes
for c in ['Narrate_Role', 'Validation_Status', 'Applies_To', 'Concept_Group', 'Value_Type', 'Measured_At',
          'Scheme_Link', 'Concern_Direction', 'Description', 'Suspicious_Pattern', 'Benign_Explanations',
          'Narrative_Caveat', 'Records_To_Review', 'Policy_Lever', 'Definition_Confidence', 'Level_Dependent']:
    feat[c] = feat['Feature_Root'].map(lambda r, c=c: MAP[r][c])
_levels = {(f, r): plain_level(f, r) for f, r in feat[['Feature', 'Feature_Root']].drop_duplicates().itertuples(index=False)}
feat['Level_Plain'] = [_levels[(f, r)][0] for f, r in zip(feat['Feature'], feat['Feature_Root'])]
feat['Level_Missing'] = [_levels[(f, r)][1] for f, r in zip(feat['Feature'], feat['Feature_Root'])]
feat['Label'] = [feature_label(r, lv) for r, lv in zip(feat['Feature_Root'], feat['Level_Plain'])]
for c in ['Description']:      # count-level rows carry {level} in their text
    feat[c] = [t.replace('{level}', lv) for t, lv in zip(feat[c], feat['Level_Plain'])]
_codes = {f: decode_feature_codes(f) for f in feat['Feature'].unique()}
feat['Code_Decoded'] = feat['Feature'].map(_codes)
feat['Is_Demographic'] = feat['Feature_Category'].isin(DEMOGRAPHIC_CATEGORIES)

# (4) Effective role: what the note may use this feature for
feat['Role'], feat['Role_Reason'] = feat['Narrate_Role'], ''
def _set_role(mask, role, reason):
    m = mask & (feat['Role_Reason'] == '')
    feat.loc[m, 'Role'] = role
    feat.loc[m, 'Role_Reason'] = reason
    prep_stats[f'role:{reason}'] += int(m.sum())
_set_role(feat['Narrate_Role'] == 'Do not narrate', 'Not narrated', 'do not narrate')
_set_role(feat['Validation_Status'] == 'Hold until confirmed', 'Not narrated', 'on hold')
_set_role(pd.Series([not applies_to(a, c) for a, c in zip(feat['Applies_To'], feat['Category'])], index=feat.index),
          'Not narrated', 'not applicable to category')
_set_role(feat['Level_Missing'], 'Not narrated', 'missing-value level')
_set_role((feat['Role'] == 'Driver') & feat['Level_Plain'].isin(NON_CONCERNING_LEVELS), 'Context only', 'level not concerning')

# (5) Provider values (Python only), for features the note may use
_needed = feat.loc[feat['Role'].isin(['Driver', 'Context only']), [provider_id, 'Category', 'Feature', 'Feature_Root']]
provider_feature_values, value_coverage = build_provider_feature_values(_needed.drop_duplicates())
feat = feat.merge(provider_feature_values, on=_k, how='left', validate='1:1')
_few = feat['n_claims'].notna() & (feat['n_claims'] < MIN_CLAIMS_FOR_VALUE)
feat.loc[_few, ['value', 'peer_median', 'peer_pct']] = np.nan
prep_stats['values_dropped_few_claims'] = int(_few.sum())
# A level-dependent measure whose strongest signal came from another analysis level: the provider-level value from the
# scoring files would describe a different calculation, so it is not quoted (and cannot confirm a scheme)
_other_level = (feat['Level_Dependent'] == 'Yes') & (feat['Best_Level'].astype(str).str.lower() != 'provider') \
    & (feat['value_source'] == 'provider_measure')
feat.loc[_other_level, ['value', 'peer_median', 'peer_pct']] = np.nan
prep_stats['values_dropped_other_analysis_level'] = int(_other_level.sum())
feat['Value_Sentence'] = [render_value_sentence(r) if role in ('Driver', 'Context only') else ''
                          for r, role in zip(feat.to_dict('records'), feat['Role'])]

# (6) A driver whose measured value sits on the harmless side of peers is background, never a reason
_harmless = (feat['Role'] == 'Driver') & feat['peer_pct'].notna() & (
    ((feat['Concern_Direction'] == 'High') & (feat['peer_pct'] < HARMLESS_PCT_HIGH))
    | ((feat['Concern_Direction'] == 'Low') & (feat['peer_pct'] > HARMLESS_PCT_LOW)))
feat.loc[_harmless, 'Role'] = 'Context only'
feat.loc[_harmless, 'Role_Reason'] = 'harmless side of peers'
prep_stats['role:harmless side of peers'] = int(_harmless.sum())

# (7) What the LLM sees: MAIN (drivers inside the 80% signal coverage), SUPPORTING (next drivers), BACKGROUND.
#     One line per Concept_Group; different codes of the same field stay separate (they are different evidence).
feat = feat.sort_values([provider_id, 'Category', 'Feature_Rank_The_Provider'], kind='stable').reset_index(drop=True)
feat['_key'] = np.where(feat['Concept_Group'] != '', feat['Concept_Group'], feat['Feature_Root'])
_code_level = (feat['Value_Type'] == 'Category level') & (feat['Feature'] != feat['Feature_Root'])
feat.loc[_code_level, '_key'] = feat.loc[_code_level, '_key'] + '|' + feat.loc[_code_level, 'Label']
feat['Shown_As'] = ''
for _, g in feat.groupby([provider_id, 'Category'], sort=False):
    used, n_main, n_sup, n_bg = set(), 0, 0, 0
    for i, r in g[g['Role'] == 'Driver'].iterrows():
        if (r['In_Main_Tier'] and n_main < MAX_MAIN and r['_key'] not in used
                and CONFIDENCE_ORDER.index(r['Definition_Confidence']) >= CONFIDENCE_ORDER.index(MIN_CONFIDENCE_FOR_MAIN)):
            feat.at[i, 'Shown_As'] = 'Main'; used.add(r['_key']); n_main += 1
    for i, r in g[(g['Role'] == 'Driver') & (feat.loc[g.index, 'Shown_As'] == '')].iterrows():
        if n_sup < MAX_SUPPORTING and r['_key'] not in used:
            feat.at[i, 'Shown_As'] = 'Supporting'; used.add(r['_key']); n_sup += 1
    for i, r in g[g['Role'] == 'Context only'].iterrows():
        if n_bg < MAX_BACKGROUND and r['_key'] not in used:
            feat.at[i, 'Shown_As'] = 'Background'; used.add(r['_key']); n_bg += 1

# (8) Patient mix: demographic share of the top-ranked signal, and whether a billing driver is among the strongest
_pk = [provider_id, 'Category']
_top = feat.groupby(_pk, sort=False).head(TOP_K_SIGNAL).copy()
_top['_demo_score'] = _top['Overall_Importance_Score'].where(_top['Is_Demographic'], 0.0)
_mix = _top.groupby(_pk)[['_demo_score', 'Overall_Importance_Score']].sum()
_mix['Patient_Mix_Share'] = (_mix['_demo_score'] / _mix['Overall_Importance_Score'].clip(lower=1e-12)).round(3)
_mix['Driver_In_Top'] = (feat.groupby(_pk, sort=False).head(PATIENT_MIX_TOP_DRIVERS)
                         .assign(_d=lambda d: d['Role'] == 'Driver').groupby(_pk)['_d'].any())
_mix['Mix_Labels'] = (_top[_top['Is_Demographic']].assign(_lab=lambda d: d['Feature_Root'].map(lambda r: MAP[r]['Replacement']))
                      .groupby(_pk)['_lab'].agg(lambda s: '; '.join(dict.fromkeys(s))))
_mix = _mix.drop(columns=['_demo_score', 'Overall_Importance_Score']).reset_index()
_mix['Mix_Labels'] = _mix['Mix_Labels'].fillna('')
_mix['Driver_In_Top'] = _mix['Driver_In_Top'].fillna(False).astype(bool)
_mix['Patient_Mix'] = (_mix['Patient_Mix_Share'] >= PATIENT_MIX_SHARE) & ~_mix['Driver_In_Top']

# (9) One row per referred provider
_counts = (feat.groupby([provider_id, 'Category'])['Shown_As'].value_counts().unstack(fill_value=0)
           .reindex(columns=['Main', 'Supporting', 'Background'], fill_value=0)
           .rename(columns=lambda c: f'n_{c.lower()}').reset_index())
providers = (referrals.merge(_counts, on=[provider_id, 'Category'], how='left')
             .merge(_mix, on=[provider_id, 'Category'], how='left'))
_has_rows = set(zip(feat[provider_id], feat['Category']))
providers['Has_Features'] = [k in _has_rows for k in zip(providers[provider_id], providers['Category'])]
for c in ['n_main', 'n_supporting', 'n_background']:
    providers[c] = providers[c].fillna(0).astype(int)
providers['Patient_Mix'] = providers['Patient_Mix'].fillna(False).astype(bool)
providers['Note_Type'] = np.select(
    [~providers['Has_Features'], providers['Patient_Mix'], providers['n_main'] == 0],
    ['evidence_only', 'patient_mix', 'evidence_only'], default='narrative_eligible')
providers['Note_Reason'] = np.select(
    [~providers['Has_Features'], providers['Patient_Mix'], providers['n_main'] == 0],
    ['no feature rows', 'patient mix', 'no billing measure in the 80% signal coverage'], default='')

features_by_provider = {k: g for k, g in feat.groupby([provider_id, 'Category'], sort=False)}
print("Feature preparation:", dict(prep_stats))
print("Value sentences:", dict(value_coverage))
print(providers.groupby('Category')['Note_Type'].value_counts().unstack(fill_value=0).to_string())

##### Narrative selection and schemes

In [ ]:
# ── Which referred providers get an LLM narrative ──
# Threshold: per-category percentile of the provider anomaly score among ALL referred providers.
# Floor and cap count only providers that can get a narrative (patient-mix and no-main-pattern providers do not
# take a slot), ranked by the same score; ties broken by provider ID so reruns select the same providers.
_thr = providers.groupby('Category')['Servicing_Provider_Anomaly_Score'].quantile(NARRATE_PCTL)
providers['Score_Threshold'] = providers['Category'].map(_thr)
providers = providers.sort_values(['Category', 'Servicing_Provider_Anomaly_Score', provider_id],
                                  ascending=[True, False, True], na_position='last', kind='stable').reset_index(drop=True)
providers['Category_Rank'] = providers.groupby('Category').cumcount() + 1
_elig = providers['Note_Type'] == 'narrative_eligible'
providers['Eligible_Rank'] = np.nan
providers.loc[_elig, 'Eligible_Rank'] = providers[_elig].groupby('Category').cumcount() + 1
providers['Narrate_Selected'] = _elig & providers['Servicing_Provider_Anomaly_Score'].notna() & (
    (providers['Servicing_Provider_Anomaly_Score'] >= providers['Score_Threshold'])
    | (providers['Eligible_Rank'] <= NARRATE_FLOOR))
if NARRATE_CAP:
    providers['Narrate_Selected'] &= providers['Eligible_Rank'] <= NARRATE_CAP
providers.loc[_elig & ~providers['Narrate_Selected'], 'Note_Type'] = 'below_cutoff'
providers.loc[providers['Note_Type'] == 'below_cutoff', 'Note_Reason'] = 'below the narrative cut-off for its category'
providers.loc[providers['Narrate_Selected'], 'Note_Type'] = 'narrative'

print(f"Selection: {NARRATE_PCTL:.0%} percentile per category, floor {NARRATE_FLOOR}, cap {NARRATE_CAP}")
print(providers.groupby('Category').agg(referred=('Note_Type', 'size'),
                                        narrative=('Narrate_Selected', 'sum'),
                                        threshold=('Score_Threshold', 'first')).round(4).to_string())
print("Note types:", providers['Note_Type'].value_counts().to_dict())


# ── Scheme labels from the dictionary (Scheme_Link of the MAIN patterns), never from the narrative ──
def derive_schemes(main_rows):
    """Returns (confirmed, possible, rationale).
    confirmed: linked schemes with the provider's measured value on the concerning side of peers (High: at or above
               SCHEME_PCT_HIGH; Low: at or below SCHEME_PCT_LOW). Only these may be named in the narrative.
    possible:  linked schemes without a measured value, or for an 'Either' measure at either extreme. Not named."""
    confirmed, possible, why = {}, {}, []
    for _, r in main_rows.iterrows():
        links = [s.strip().lower() for s in str(r['Scheme_Link']).split('|')
                 if s.strip() and s.strip().lower() not in ('none', 'other')]
        if not links:
            continue
        pct, d = r.get('peer_pct'), r['Concern_Direction']
        measured = pct is not None and not pd.isna(pct)
        if measured and ((d == 'High' and pct >= SCHEME_PCT_HIGH) or (d == 'Low' and pct <= SCHEME_PCT_LOW)):
            target, how = confirmed, f"{pct_ordinal(pct)} percentile"
        elif measured and d == 'Either' and (pct >= SCHEME_PCT_HIGH or pct <= SCHEME_PCT_LOW):
            target, how = possible, f"{pct_ordinal(pct)} percentile, direction not defined"
        elif not measured:
            target, how = possible, 'no provider value'
        else:
            continue
        for lab in links:
            target.setdefault(lab, []).append(r['Label'])
        why.append(f"{r['Label']} ({how})")
    for lab in [l for l in confirmed if l in SCHEMES_NEEDING_TWO_MEASURES and len(set(confirmed[l])) < 2]:
        possible.setdefault(lab, []).extend(confirmed.pop(lab))      # one measure alone: possible, not confirmed
    for lab in confirmed:
        possible.pop(lab, None)
    return list(confirmed), list(possible), '; '.join(why)


def scheme_columns(key, note_type):
    """Detected_Schemes, Possible_Schemes, Scheme_Rationale, allowed scheme names for the narrative."""
    if note_type == 'patient_mix':
        return 'Not assessed (patient mix)', '', '', []
    g = features_by_provider.get(key)
    main = g[g['Shown_As'] == 'Main'] if g is not None else None
    if main is None or main.empty:
        return 'Not assessed (no billing measure to narrate)', '', '', []
    confirmed, possible, why = derive_schemes(main)
    any_link = any(str(s).strip().lower() not in ('', 'none') for s in main['Scheme_Link'])
    if confirmed:
        detected = ', '.join(confirmed)
    elif possible:
        detected = 'Not confirmed (see Possible_Schemes)'
    elif not any_link:
        detected = NEW_PATTERN_LABEL
    else:
        detected = 'No scheme confirmed (linked measures not on the concerning side)'
    return detected, ', '.join(possible), why, [s for s in confirmed if s in SCHEME_DEFINITIONS_BY_LABEL]

##### Prompt and deterministic notes

In [ ]:
# ── Prompt: only facts computed above; scheme names limited to the confirmed ones ──
_SCHEME_CLAUSE = re.compile(r'[,;]?\s*consistent with [^.;]*|\s*\(possible [^)]*\)', re.I)


def _sentence(s):
    s = str(s or '').strip()
    return s if not s or s.endswith(('.', '!', '?')) else s + '.'


_SCHEME_NAMES = sorted([s for s in KNOWN_SCHEMES if s != 'other'] + ['upcod', 'unbundl', 'phantom', 'duplicate claim'],
                       key=len, reverse=True)


def _without_scheme_sentences(text, allowed):
    keep = [s for s in re.split(r'(?<=[.;])\s+', str(text or ''))
            if not any(n in s.lower() and not any(n in a for a in allowed) for n in _SCHEME_NAMES)]
    return ' '.join(keep).strip()


def _pattern_line(k, r, full, allowed=()):
    line = f"  {k}. {r['Label']}"
    if r.get('Level_Dependent') == 'Yes' and str(r.get('Best_Level', '')).lower() in LEVEL_PHRASE:
        line += f" ({LEVEL_PHRASE[str(r['Best_Level']).lower()]})"
    if r['Code_Decoded']:
        line += f" [{r['Code_Decoded']}]"
    parts = [_sentence(r['Description']), _sentence(r['Value_Sentence'])]
    if full:
        suspicious = _SCHEME_CLAUSE.sub('', r['Suspicious_Pattern']).strip()
        benign = _without_scheme_sentences(r['Benign_Explanations'], allowed)
        caveat = _without_scheme_sentences(r['Narrative_Caveat'], allowed)
        parts += [_sentence(_without_scheme_sentences(suspicious, allowed)),
                  _sentence(f"Other explanations: {benign}") if benign else '',
                  _sentence(f"Note: {caveat}") if caveat else '']
    return line + ': ' + ' '.join(p for p in parts if p)


def build_prompt(pid, cat, g, allowed_schemes):
    main = g[g['Shown_As'] == 'Main']
    sup = g[g['Shown_As'] == 'Supporting']
    bg = g[g['Shown_As'] == 'Background']
    if allowed_schemes:
        scheme_rule = ("- You may say a main pattern is 'consistent with' one of these schemes, and only these: "
                       + '; '.join(allowed_schemes) + ". Never say a scheme happened or was intended.\n")
        scheme_defs = "SCHEME DEFINITIONS:\n" + '\n'.join(f"- {s}: {SCHEME_DEFINITIONS_BY_LABEL[s]}"
                                                           for s in allowed_schemes) + '\n'
    else:
        scheme_rule = "- Do not name any billing scheme; describe the patterns only.\n"
        scheme_defs = ''
    system_msg = (
        "You write short case notes for Medicaid program-integrity investigators. "
        f"Reasoning: {REASONING_LEVEL}\n\n"
        "RULES:\n"
        "- Use only facts and numbers in the user message. Do not calculate new numbers, ratios, multiples or "
        "differences, and do not round a number into a different one.\n"
        "- All figures come from PAID claims: write 'paid'. Paid is not proof that a service did or did not happen; "
        "never say a service was or was not delivered, provided or real.\n"
        "- Say the provider is higher or lower than peers only when the message gives a provider value with a peer "
        "median or percentile. Otherwise say the pattern was unusual compared with peers.\n"
        "- Describe the MAIN PATTERNS in order (at most 3). SUPPORTING items may add detail to them. BACKGROUND items "
        "may be mentioned once, introduced with 'In addition,', and never as a reason for the flag.\n"
        "- Where an 'Other explanations' note plausibly applies, mention it in one short clause.\n"
        + scheme_rule +
        "- Code descriptions only identify codes. Do not judge medical necessity, quality of care, or whether a "
        "diagnosis supports, fits or justifies a service.\n"
        "- No intent words, recommendations or next steps. No headings, lists or bold text.\n"
        "- Never use these words: score, model, anomaly, outlier, z-score, algorithm, fraud, billed.\n"
        "- No intensifiers such as markedly, significantly, dramatically or extremely.\n"
        "- 100 to 180 words in 2 short paragraphs.\n"
        f"- The first sentence must begin exactly: Provider {pid} was flagged because its {cat} paid claims\n\n"
        + scheme_defs)
    profile = []
    services = main_services(pid, cat)
    if services:
        profile.append(f"  Main services: {services}")
    peers = peer_comparison_lines(pid, cat)
    user_msg = (
        "Write a case note for this provider.\n\n"
        f"PROVIDER: {pid} | CLAIM CATEGORY: {cat}\n\n"
        + ("PROVIDER PROFILE:\n" + '\n'.join(profile) + '\n\n' if profile else '')
        + ('PEER COMPARISON:\n' + '\n'.join(peers) + '\n\n' if peers else '')
        + "MAIN PATTERNS (the reasons the provider stood out, strongest first):\n"
        + '\n'.join(_pattern_line(k + 1, r, True, allowed_schemes) for k, (_, r) in enumerate(main.iterrows())) + '\n\n'
        + "SUPPORTING PATTERNS (may add detail to the main patterns):\n"
        + ('\n'.join(_pattern_line(k + 1, r, False, allowed_schemes) for k, (_, r) in enumerate(sup.iterrows())) or '  None.') + '\n\n'
        + "BACKGROUND (never a reason for the flag):\n"
        + ('\n'.join(f"  {k + 1}. {r['Label']}" + (f": {r['Value_Sentence']}" if r['Value_Sentence'] else '')
                     for k, (_, r) in enumerate(bg.iterrows())) or '  None.') + '\n')
    return [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': user_msg}], bool(peers)


# ── Deterministic notes (no model output) ──
def evidence_only_note(pid, cat, reason, g=None):
    text = f"Provider {pid} was flagged because its {cat} paid claims differed from peers. "
    if reason == 'below the narrative cut-off for its category':
        text += ("It ranked below this run's narrative cut-off for its claim category, so no written case note was "
                 "produced; the measures behind the flag are on the evidence card.")
    elif reason == 'generation failed':
        text += "A written case note could not be produced in this run; the measures behind the flag are on the evidence card."
    elif reason == 'no feature rows':
        text += "No explaining measures were available for this provider in this run."
    else:
        text += ("None of the strongest measures behind the flag describes a billing pattern that can be narrated yet "
                 "(they are background measures or have definitions awaiting confirmation). The measures are listed on "
                 "the evidence card.")
    return text


def patient_mix_note(pid, cat, labels):
    lab = f" ({labels.lower()})" if labels else ''
    return (f"Provider {pid} was flagged in {cat} mainly because its patient mix differs from peers{lab}. "
            "Patient mix alone is not a billing concern. None of the strongest measures behind the flag describes how "
            "the provider bills, so this provider is listed for model review. All measures are on the evidence card.")

##### Generation

In [ ]:
# ── Generation with GPT-OSS (Harmony channels; the hidden reasoning is never published) ──
def _harmony_text(content):
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return ''.join(_harmony_text(c) for c in content)
    text = getattr(content, 'text', None)
    return text if isinstance(text, str) else ''


def _parse_completion(new_token_ids, max_new_tokens):
    """(final_text, analysis_chars, meta) from generated token ids."""
    meta = {'new_tokens': len(new_token_ids), 'hit_token_limit': len(new_token_ids) >= max_new_tokens}
    final_text, analysis = '', ''
    try:
        from openai_harmony import Role
        for msg in harmony_enc.parse_messages_from_completion_tokens(new_token_ids, Role.ASSISTANT):
            channel = str(getattr(msg, 'channel', '') or '')
            text = _harmony_text(getattr(msg, 'content', None))
            if channel == 'final':
                final_text = text
            elif channel == 'analysis':
                analysis += text
        meta['parse'] = 'harmony'
    except Exception as e:
        meta['parse'] = f'fallback ({type(e).__name__})'
        raw = tokenizer.decode(new_token_ids, skip_special_tokens=False)
        if '<|channel|>final<|message|>' in raw:
            final_text = raw.split('<|channel|>final<|message|>')[-1]
            for stop in ['<|return|>', '<|end|>', '<|start|>']:
                final_text = final_text.split(stop)[0]
    meta['final_found'] = bool(final_text.strip())
    return final_text.strip(), len(analysis), meta


def _stop_token_ids():
    try:
        return set(harmony_enc.stop_tokens_for_assistant_actions())
    except Exception:
        return set()


def generate_batch(message_lists, max_new_tokens):
    """One model.generate call for a list of conversations. Returns [(final_text, analysis_chars, meta)]."""
    if len(message_lists) == 1:
        inputs = tokenizer.apply_chat_template(message_lists[0], return_tensors='pt', add_generation_prompt=True,
                                               return_dict=True, reasoning_effort=REASONING_LEVEL)
        input_ids = inputs['input_ids'].to(model.device)
        attention_mask = inputs.get('attention_mask')
        attention_mask = attention_mask.to(model.device) if attention_mask is not None else None
        with torch.no_grad():
            out = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=False)
        return [_parse_completion(out[0][input_ids.shape[1]:].tolist(), max_new_tokens)]
    # Batched: left padding so every prompt ends at the same position
    seqs = []
    for msgs in message_lists:
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True,
                                            reasoning_effort=REASONING_LEVEL)['input_ids']
        seqs.append(list(ids[0]) if ids and isinstance(ids[0], (list, tuple)) else list(ids))
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    width = max(len(s) for s in seqs)
    input_ids = torch.tensor([[pad] * (width - len(s)) + s for s in seqs]).to(model.device)
    attention_mask = torch.tensor([[0] * (width - len(s)) + [1] * len(s) for s in seqs]).to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=pad)
    stops, results = _stop_token_ids(), []
    for row in out:
        new = row[width:].tolist()
        cut = next((i for i, t in enumerate(new) if t in stops), None)
        if cut is not None:
            new = new[:cut + 1]
        results.append(_parse_completion(new, max_new_tokens))
    return results


def _is_cut_off(text, meta):
    return meta.get('hit_token_limit') and not normalize_text(text).rstrip('"\')').endswith(('.', '!', '?'))


def write_notes(jobs):
    """jobs: list of dicts with 'messages'. Adds raw_final, attempts, new_tokens, hit_token_limit, parse, error.
    One retry with a larger budget when the answer is missing or cut off; on a missing answer the retry also asks
    for the final note directly (a greedy retry of the same prompt would repeat the same output)."""
    pending = list(jobs)
    for attempt, budget in enumerate((MAX_NEW_TOKENS, RETRY_MAX_NEW_TOKENS), start=1):
        if not pending:
            break
        retry = []
        for i in range(0, len(pending), max(GEN_BATCH_SIZE, 1)):
            chunk = pending[i:i + max(GEN_BATCH_SIZE, 1)]
            msgs = []
            for j in chunk:
                m = j['messages']
                if attempt > 1 and not j.get('hit_token_limit'):
                    m = m[:-1] + [{'role': 'user', 'content': m[-1]['content'] + "\nWrite only the final case note now."}]
                msgs.append(m)
            try:
                results = generate_batch(msgs, budget)
            except Exception as e:
                if len(chunk) > 1:          # e.g. out of memory: fall back to one at a time
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
                    results = []
                    for m in msgs:
                        try:
                            results.extend(generate_batch([m], budget))
                        except Exception as e1:
                            results.append(('', 0, {'error': f'{type(e1).__name__}: {e1}'[:300]}))
                else:
                    results = [('', 0, {'error': f'{type(e).__name__}: {e}'[:300]})]
            for j, (text, a_chars, meta) in zip(chunk, results):
                j.update(raw_final=text, attempts=attempt, analysis_chars=a_chars,
                         new_tokens=meta.get('new_tokens', 0), hit_token_limit=bool(meta.get('hit_token_limit')),
                         parse=meta.get('parse', ''), error=meta.get('error', ''))
                if (not text or _is_cut_off(text, meta)) and attempt == 1:
                    retry.append(j)
        pending = retry
    return jobs


_SENT = re.compile(r'(?<=[.!?])(?<!\bvs\.)(?<!e\.g\.)(?<!i\.e\.)(?<!approx\.)\s+(?=[A-Z"(])')


def clean_note(text):
    """Normalize characters; drop markdown, headings and repeated sentences; keep paragraphs."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = normalize_text(text)
    text = re.sub(r'<\|[^|]+\|>', '', text)
    text = re.sub(r'(?m)^[ \t]{0,3}(#{1,6}[ \t]*|[-*•][ \t]+|\d+[.)][ \t]+)', '', text)
    text = text.replace('**', '').replace('__', '')
    text = re.sub(r'(?im)^[ \t]*(summary|case note|answer)[ \t]*:[ \t]*', '', text)
    paras, seen = [], set()
    for para in re.split(r'\n\s*\n', text):
        kept = []
        for s in _SENT.split(para.strip()):
            key = s.strip().lower().rstrip('.')
            if key and key not in seen:
                seen.add(key)
                kept.append(s.strip())
        if kept:
            paras.append(' '.join(kept))
    return re.sub(r'[ \t]+', ' ', '\n\n'.join(paras)).strip()

##### Quality checks

In [ ]:
# ── QA: three checks (numbers grounded in the prompt, banned words, opening and category) plus length ──
QA_BANNED = [
    (r'\bscor(e|es|ed|ing)\b', 'score'), (r'\bz-?scores?\b', 'z-score'), (r'\banomal', 'anomaly'),
    (r'\boutliers?\b', 'outlier'), (r'\bmodel(s|ed|ing)?\b', 'model'), (r'\bshap\b', 'shap'),
    (r'\balgorithm', 'algorithm'), (r'machine learning|autoencoder|isolation forest', 'ml term'),
    (r'\bbilled\b', 'billed'), (r'\bfraud', 'fraud'),
    (r'\b(markedly|significantly|dramatically|substantially|extremely|grossly|alarmingly|egregious(ly)?)\b', 'intensifier'),
    (r'\b(deliberate(ly)?|intentional(ly)?|knowingly|purpose(ly|fully)|to maximi[sz]e|in order to (increase|inflate|obtain)'
     r'|gam(e|ed|ing) the|exploit(s|ed|ing)?)\b', 'intent'),
    (r'medical(ly)? necess|clinically (justif|appropriate|necessary|indicated)|\bunjustified\b|\bunnecessary\b'
     r'|\b(does|did|do) not (support|fit|match|justify)\b|\binconsistent with (the |a |its )?diagnos'
     r'|\bnot (supported|indicated|justified) by\b|clinically (unwarranted|unnecessary|inappropriate|unsupported)'
     r'|\bwould not (ordinarily |normally |usually )?(require|need)|\bnot (medically )?(required|warranted)\b', 'clinical judgment'),
    (r'\b(was|were|been) (not|never) (actually )?(delivered|provided|rendered|performed|received)\b'
     r'|\bnot (actually )?(delivered|provided|rendered|received|performed)\b|\bnever (happened|occurred|took place)\b'
     r'|\bdid not (actually )?(happen|occur|take place)\b|\b(fake|fictitious|phony|bogus|fabricated)\b'
     r'|\bmay (not|never) have (taken place|occurred|happened)\b|\bno (evidence|record|sign) (that )?(the )?\w+ (occurred|took place|happened)\b',
     'delivery claim'),
    (r'\brecommend|should (be )?(investigat|review|audit)|\bwarrants?\b|further (review|investigation)|next steps?\b',
     'recommendation'),
    (r'\btwice (as|the)\b|\b(double|triple|quadruple) (the|that)\b|\b\d+(\.\d+)?\s*(times|-fold|x)\s+(the|as|higher|more|that|what|peers?)\b'
     r'|\b(two|three|four|five|six|ten) times\b|\b(half|a third|one third|a quarter|two thirds) (of |as |the )',
     'ratio'),
]
SCHEME_SYNONYMS = [
    (r'double[- ]?bill|paid twice|duplicate (claims|payments)', 'duplicate billing'),
    (r'\bup-?\s?cod(e|ed|ing)\b', 'upcoding'), (r'\bphantom\b', 'phantom billing'), (r'\bunbundl', 'unbundling'),
    (r'inflated (mileage|miles)|padded (mileage|miles|wait)', 'mileage inflation'), (r'\bkickbacks?\b', 'kickback'),
]
_NUM_RE = re.compile(r'(?<![A-Za-z0-9_.])-?\$?\d[\d,]*(?:\.\d+)?(?:%|st|nd|rd|th)?(?![A-Za-z0-9_])')


def _numbers(text):
    out = []
    for m in _NUM_RE.finditer(normalize_text(text)):
        tok = m.group(0)
        core = re.sub(r'[^\d.\-]', '', tok.replace(',', '')).rstrip('.')
        try:
            val = float(core)
        except ValueError:
            continue
        after = normalize_text(text)[m.end():m.end() + 9].lower()
        pct_word = after.startswith(' percent') and not after.startswith(' percentil')
        kind = '$' if '$' in tok else '%' if (tok.endswith('%') or pct_word) else 'ord' if re.search(r'(st|nd|rd|th)$', tok) else 'n'
        out.append((tok, val, kind))
    return out


def quality_check(note, pid, cat, prompt_user, allowed_schemes):
    issues, s = [], normalize_text(note)
    words = len(s.split())
    if words < 40:
        issues.append('too_short')
    if words > 240:
        issues.append('too_long')
    if s and not s.rstrip().rstrip('"\')').endswith(('.', '!', '?')):
        issues.append('truncated')
    # (1) Banned words, after removing the scheme names the note may use
    low = s.lower()
    for label in sorted([k for k in KNOWN_SCHEMES if k != 'other'], key=len, reverse=True):
        if label in low:
            if label not in allowed_schemes:
                issues.append(f'unsupported_scheme:{label}')
            low = low.replace(label, ' ')
    for pat, label in SCHEME_SYNONYMS:
        m = re.search(pat, low)
        if m and label not in allowed_schemes:
            issues.append(f'unsupported_scheme:{label}({m.group(0)})')
        low = re.sub(pat, ' ', low)
    for label in allowed_schemes:            # an allowed scheme may only be named as 'consistent with'
        for m in re.finditer(re.escape(label), s.lower()):
            if 'consistent with' not in s.lower()[max(0, m.start() - 45):m.start()]:
                issues.append(f'scheme_as_fact:{label}')
                break
    for pat, name in QA_BANNED:
        m = re.search(pat, low)
        if m:
            issues.append(f'banned:{name}({m.group(0)})')
    # (2) Every number must appear in the prompt as the same kind ($, %, ordinal, plain); list numbering ignored
    p_text = re.sub(r'(?m)^\s*\d+\.\s', ' ', normalize_text(prompt_user))
    p_items = [(v, k) for _, v, k in _numbers(p_text)]

    def _grounded(v, k):     # exact, or a prompt value of 10 or more rounded half up to a whole number
        return any(k == pk and (v == pv or (abs(pv) >= 10 and v == math.floor(pv + 0.5))) for pv, pk in p_items)
    bad = sorted({tok for tok, v, k in _numbers(s) if not _grounded(v, k)})
    if bad:
        issues.append(f"ungrounded_number:{'|'.join(bad[:5])}")
    # A scheme definition (system message only) repeated as if it were a finding about this provider
    user_words = _norm_words(prompt_user)
    for label in allowed_schemes:
        w = re.sub(r'[^a-z0-9 ]+', ' ', SCHEME_DEFINITIONS_BY_LABEL.get(label, '').lower()).split()
        for i in range(len(w) - 5):
            frag = ' ' + ' '.join(w[i:i + 6]) + ' '
            if frag in _norm_words(s) and frag not in user_words:
                issues.append(f'definition_as_finding:{label}')
                break
    # Provider value and peer median must not trade places ('provider value (peer median X' pairs from the prompt)
    for pv, pmed in re.findall(r'(\$?[\d,]+(?:\.\d+)?%?)[^()\n]{0,80}\(peer median (\$?[\d,]+(?:\.\d+)?%?)', normalize_text(prompt_user)):
        if pv != pmed and (re.search(rf'median (?:of )?(?:about |around )?{re.escape(pv)}(?![\d,])', s)
                           or re.search(rf'{re.escape(pmed)}(?![\d,])[^.]{{0,40}}against a peer median', s)):
            issues.append(f'swapped_value:{pv}/{pmed}')
    # (3) Opening sentence and category
    first = (_SENT.split(s.strip()) or [''])[0]
    if not re.match(rf'\s*Provider\s+{re.escape(str(pid))}\s+was flagged because its\b', first):
        issues.append('opening_template')
    if not category_mentioned(first, cat):
        issues.append('category_not_in_opening')
    return issues or ['pass']

##### Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai_harmony import load_harmony_encoding, HarmonyEncodingName

_n_llm = int(providers['Narrate_Selected'].sum())
if _n_llm == 0:
    print("No provider selected for a narrative; the model is not loaded.")
    tokenizer = model = harmony_enc = None
else:
    _hf_token = os.environ.get('HF_TOKEN')
    if not _hf_token and HF_SECRET_SCOPE:
        _hf_token = dbutils.secrets.get(scope=HF_SECRET_SCOPE, key=HF_SECRET_KEY)
    if _hf_token:
        login(_hf_token)
    print(f"Loading {MODEL_ID} for {_n_llm} narratives")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if GEN_BATCH_SIZE > 1:
        tokenizer.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto", trust_remote_code=True)
    harmony_enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
    print(f"Model loaded; device map: {getattr(model, 'hf_device_map', 'auto')}")

##### Write the notes
Deterministic notes first; narratives with checkpoint and resume.

In [ ]:
# ── Write the notes ──
def _base_row(p):
    key = (p[provider_id], p['Category'])
    detected, possible, why, allowed = scheme_columns(key, p['Note_Type'])
    g = features_by_provider.get(key)
    main = g[g['Shown_As'] == 'Main'] if g is not None else None
    return {
        provider_id: p[provider_id], 'Category': p['Category'], 'LLM_Summary': '',
        'Detected_Schemes': detected, 'Scheme_Rationale': why, 'QA_Flags': '', 'QA_Status': '',
        'Baselines_Available': False, 'Scoring_Date': scoring_date,
        'Narrative_Features': ' | '.join(main['Label']) if main is not None else '',
        'Main_Pattern_Confidence': (min(main['Definition_Confidence'], key=CONFIDENCE_ORDER.index)
                                    if main is not None and len(main) else ''),
        'Notebook_Version': NOTEBOOK_VERSION, 'Dictionary_Version': DICTIONARY_VERSION,
        'Possible_Schemes': possible, 'Note_Type': p['Note_Type'], 'Note_Reason': p['Note_Reason'],
        'Provider_Anomaly_Score': p['Servicing_Provider_Anomaly_Score'], 'Category_Rank': p['Category_Rank'],
        'Narrate_Selected': bool(p['Narrate_Selected']), 'Patient_Mix_Share': p.get('Patient_Mix_Share'),
        'Provider_Perspective_Score': p['Provider_Perspective_Score'],
        'Patient_Perspective_Score': p['Patient_Perspective_Score'],
        'Visit_Perspective_Score': p['Visit_Perspective_Score'], 'Claim_Perspective_Score': p['Claim_Perspective_Score'],
    }, allowed


def _prompt_sha(messages):
    return hashlib.sha1('\n'.join(m['content'] for m in messages).encode()).hexdigest()


# Resume: a narrative is reused only when it was written for exactly the same prompt (same features, values,
# dictionary text and settings) and did not fail; anything else is generated again.
done, _bad_lines = {}, 0
if RESUME and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:          # last line cut off by a crash
                _bad_lines += 1
                continue
            if rec['summary'].get('QA_Status') != 'generation_failed':
                done[(str(rec['summary'][provider_id]), rec['summary']['Category'])] = rec
    print(f"Checkpoint: {len(done)} narratives available for reuse" + (f"; {_bad_lines} unreadable lines skipped" if _bad_lines else ''))

summary_rows, audit_rows, jobs, reused = [], [], [], 0
for p in providers.to_dict('records'):
    key = (p[provider_id], p['Category'])
    row, allowed = _base_row(p)
    if p['Note_Type'] == 'patient_mix':
        row.update(LLM_Summary=patient_mix_note(p[provider_id], p['Category'], p.get('Mix_Labels', '')),
                   QA_Flags='patient_mix', QA_Status='patient_mix')
        summary_rows.append(row)
    elif p['Note_Type'] in ('evidence_only', 'below_cutoff'):
        row.update(LLM_Summary=evidence_only_note(p[provider_id], p['Category'], p['Note_Reason']),
                   QA_Flags=p['Note_Type'], QA_Status='evidence_only')
        summary_rows.append(row)
    else:
        messages, has_peers = build_prompt(p[provider_id], p['Category'], features_by_provider[key], allowed)
        row['Baselines_Available'] = has_peers
        prev = done.get(key)
        if prev is not None and prev['audit'].get('Prompt_SHA1') == _prompt_sha(messages):
            flags = quality_check(prev['summary']['LLM_Summary'], p[provider_id], p['Category'],
                                  messages[1]['content'], allowed)                       # QA rules may have changed
            row.update(LLM_Summary=prev['summary']['LLM_Summary'], QA_Flags=', '.join(flags),
                       QA_Status='pass' if flags == ['pass'] else 'needs_review')
            summary_rows.append(row)
            audit_rows.append(prev['audit'])
            reused += 1
        else:
            jobs.append({'row': row, 'allowed': allowed, 'messages': messages})

print(f"Deterministic notes: {len(summary_rows) - reused} | reused from checkpoint: {reused} | to generate: {len(jobs)}")

_chunk = max(GEN_BATCH_SIZE, 1) * 5
for start in range(0, len(jobs), _chunk):
    batch = write_notes(jobs[start:start + _chunk])
    with open(CHECKPOINT_FILE, 'a') as ckpt:
        for j in batch:
            row, pid, cat = j['row'], j['row'][provider_id], j['row']['Category']
            if j.get('raw_final'):
                note = clean_note(j['raw_final'])
                flags = quality_check(note, pid, cat, j['messages'][1]['content'], j['allowed'])
                row.update(LLM_Summary=note, QA_Flags=', '.join(flags),
                           QA_Status='pass' if flags == ['pass'] else 'needs_review')
            else:
                row.update(LLM_Summary=evidence_only_note(pid, cat, 'generation failed'),
                           QA_Flags='generation_failed' + (f": {j['error']}" if j.get('error') else ''),
                           QA_Status='generation_failed', Note_Type='generation_failed')
            audit = {provider_id: pid, 'Category': cat, 'Scoring_Date': scoring_date, 'Note_Type': row['Note_Type'],
                     'Attempts': j.get('attempts', 0), 'New_Tokens': j.get('new_tokens', 0),
                     'Hit_Token_Limit': j.get('hit_token_limit', False), 'Parse': j.get('parse', ''),
                     'Error': j.get('error', ''), 'Analysis_Chars': j.get('analysis_chars', 0),
                     'Prompt_System': j['messages'][0]['content'], 'Prompt_User': j['messages'][1]['content'],
                     'Raw_Final': j.get('raw_final', ''), 'Prompt_SHA1': _prompt_sha(j['messages'])}
            summary_rows.append(row)
            audit_rows.append(audit)
            ckpt.write(json.dumps({'summary': row, 'audit': audit}, default=str) + '\n')
    print(f"  generated {min(start + _chunk, len(jobs))}/{len(jobs)}")

output_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)
print("\nNotes:", output_df['QA_Status'].value_counts().to_dict())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"Narratives passing QA: {(_narr['QA_Status'] == 'pass').mean():.0%} of {len(_narr)}")

##### Save

In [ ]:
# ── Save ──
# Same summary file names as v3.4, so downstream reports keep working; new columns are added at the end.
output_df.to_csv(f'{OUTPUT_DIR}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony.csv', index=False)
output_df.to_csv(f'{OUTPUT_DIR}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony_{scoring_date}.csv',
                 index=False)
audit_df.to_json(f'{OUTPUT_DIR}/llm_summary_audit_{scoring_date}.jsonl', orient='records', lines=True)

# Evidence card input: every feature of every referred provider, with what the note did with it
features_out = feat[[provider_id, 'Category', 'Feature', 'Feature_Root', 'Label', 'Feature_Category', 'Best_Level',
                     'Levels_In_File', 'Analysis_Levels_In_Agreement', 'Importance_Tier', 'In_Main_Tier',
                     'Feature_Rank_The_Provider', 'Overall_Importance_Score', 'Narrate_Role', 'Validation_Status',
                     'Role', 'Role_Reason', 'Shown_As', 'Value_Sentence', 'value', 'peer_median', 'peer_pct',
                     'Scheme_Link', 'Code_Decoded', 'Records_To_Review', 'Policy_Lever', 'Definition_Confidence',
                     'Level_Dependent']]
features_out.to_csv(f'{OUTPUT_DIR}/llm_summary_features_{scoring_date}.csv', index=False)

# Model-review list: providers flagged mainly for patient mix
_pm = providers[providers['Note_Type'] == 'patient_mix'][[provider_id, 'Category', 'Servicing_Provider_Anomaly_Score',
                                                          'Patient_Mix_Share', 'Mix_Labels', 'Patient_Perspective_Score',
                                                          'Provider_Perspective_Score', 'Visit_Perspective_Score',
                                                          'Claim_Perspective_Score']]
_pm.to_csv(f'{OUTPUT_DIR}/patient_mix_providers_{scoring_date}.csv', index=False)
print(f"Saved {len(output_df)} notes, {len(audit_df)} audit records, {len(features_out)} feature rows, "
      f"{len(_pm)} patient-mix providers.")

##### Save to Delta table

In [ ]:
# Delta write (uncomment to run). Columns are selected by name; mergeSchema adds the v4 columns to an older table.
# spark = SparkSession.builder.getOrCreate()
# _cols = [provider_id, 'Category', 'LLM_Summary', 'Detected_Schemes', 'Scheme_Rationale', 'QA_Flags', 'QA_Status',
#          'Baselines_Available', 'Scoring_Date', 'Narrative_Features', 'Notebook_Version', 'Dictionary_Version',
#          'Possible_Schemes', 'Note_Type', 'Note_Reason', 'Provider_Anomaly_Score', 'Category_Rank', 'Narrate_Selected',
#          'Patient_Mix_Share']
# schema = StructType([StructField(c, StringType(), True) for c in _cols])
# _out = output_df[_cols].astype(str)
# if not _out.empty:
#     df_spark = spark.createDataFrame(_out, schema=schema)
#     if not spark.catalog.tableExists(test_table_llm_summarization):
#         df_spark.write.mode('overwrite').saveAsTable(test_table_llm_summarization)
#     else:
#         df_spark.write.mode('append').option('mergeSchema', 'true').saveAsTable(test_table_llm_summarization)

##### Trust report

In [ ]:
# ── Trust report: read before sharing the notes ──
print("=" * 72)
print(f"Run {NOTEBOOK_VERSION} | dictionary {DICTIONARY_VERSION} | scoring date {scoring_date} | referrals: {REFERRAL_SOURCE}")
print(f"Referred provider-categories: {len(providers)} | in importance file but not referred (skipped): "
      f"{load_stats['importance_only_not_referred']} | referred without feature rows: {load_stats['referred_without_feature_rows']}")
print("Note types:", output_df['Note_Type'].value_counts().to_dict())
print("QA status:", output_df['QA_Status'].value_counts().to_dict())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"  narratives passing QA: {(_narr['QA_Status'] == 'pass').mean():.0%} of {len(_narr)}")
    _fc = Counter(f.split(':')[0] + (':' + f.split(':')[1].split('(')[0] if f.startswith('banned:') else '')
                  for flags in _narr['QA_Flags'] for f in flags.split(', ') if f and f != 'pass')
    print("  QA flags:", dict(_fc.most_common()))
print(f"Feature roles: {feat['Role'].value_counts().to_dict()}")
print(f"  reasons: {feat.loc[feat['Role_Reason'] != '', 'Role_Reason'].value_counts().to_dict()}")
_shown = feat[feat['Shown_As'].isin(['Main', 'Supporting'])]
if len(_shown):
    print(f"  patterns shown to the LLM with a Python-computed value: {(_shown['Value_Sentence'] != '').mean():.0%}")
print("  value sources:", dict(value_coverage))
if len(feat[feat['Shown_As'] == 'Main']):
    print("  main patterns by Definition_Confidence:", feat.loc[feat['Shown_As'] == 'Main', 'Definition_Confidence'].value_counts().to_dict())
if 'ENC_CLM_IND' in df_scoring_master.columns:     # encounter lines: services paid by a health plan, reported to AHCCCS
    print("  ENC_CLM_IND by category (check how PMT_AMT is filled for encounters):")
    print(df_scoring_master.groupby('Category')['ENC_CLM_IND'].value_counts(dropna=False).unstack(fill_value=0).to_string())
print(f"Patient mix providers (model review list): {int((providers['Note_Type'] == 'patient_mix').sum())} "
      f"{providers.loc[providers['Note_Type'] == 'patient_mix', 'Category'].value_counts().to_dict()}")
print("Detected schemes (narratives):", output_df.loc[output_df['Note_Type'] == 'narrative', 'Detected_Schemes']
      .value_counts().head(10).to_dict())
if len(audit_df):
    print(f"Generations needing a retry: {int((audit_df['Attempts'] > 1).sum())}; failed: "
          f"{int((output_df['QA_Status'] == 'generation_failed').sum())}; median new tokens: {audit_df['New_Tokens'].median():.0f}")
if _unknown_tiers:
    print(f"Importance_Tier values outside {sorted(MAIN_TIERS)} (never main patterns): {_unknown_tiers}")
if UNKNOWN_LEVELS:
    print(f"Untranslated tier labels (shown as 'an unusual level'): {sorted(UNKNOWN_LEVELS)[:10]}")
_seen = set(_shown['Feature']) if len(_shown) else set()
_no_desc = sorted({c for c in CODES_WITHOUT_DESCRIPTION if any(c in f for f in _seen)})
_grp = sorted({c for c in CODES_DESCRIBED_BY_GROUP if any(c in f for f in _seen)})
print(f"Codes shown with only their ICD-10 group: {len(_grp)} {_grp[:5]} | with no description: {len(_no_desc)} {_no_desc[:8]}")
print("Share 'pass', 'evidence_only' and 'patient_mix' notes; read 'needs_review' and 'generation_failed' first.")
print("=" * 72)